Import libraries and load datasets

In [7]:
import pandas as pd
import numpy as np
import csv

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

talks=pd.read_csv('/content/cleaned_ted_talks.csv', quoting=csv.QUOTE_NONE, on_bad_lines='skip')

print(talks.shape)

talks.head()

(1574, 18)


,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,comments,description,duration,event,film_date,languages,main_speaker,name,num_speaker,published_date,ratings,related_talks,speaker_occupation,tags,title,url,views,transcript
4553,Sir Ken Robinson makes an entertaining and profoundly moving case for creating an education system that nurtures (rather than undermines) creativity.,1164,TED2006,1140825600,60,Ken Robinson,Ken Robinson: Do schools kill creativity?,1,1151367060,"""[{'id': 7",'name': 'Funny','count': 19645},{'id': 1,'name': 'Beautiful','count': 4573},{'id': 9,'name': 'Ingenious','count': 6073},{'id': 3,'name': 'Courageous','count': 3253},{'id': 11,'name': 'Longwinded','count': 387},{'id': 2,'name': 'Confusing','count': 242},{'id': 8,'name': 'Informative','count': 7346},{'id': 22,'name': 'Fascinating','count': 10581},{'id': 21,'name': 'Unconvincing','count': 300},{'id': 24,'name': 'Persuasive','count': 10704},{'id': 23,'name': 'Jaw-dropping','count': 4439},{'id': 25,'name': 'OK','count': 1174},{'id': 26,'name': 'Obnoxious','count': 209},{'id': 10,'name': 'Inspiring',"'count': 24924}]""","""[{'id': 865",'hero': 'https://pe.tedcdn.com/images/ted/172559_800x600.jpg','speaker': 'Ken Robinson','title': 'Bring on the learning revolution!','duration': 1008,'slug': 'sir_ken_robinson_bring_on_the_revolution','viewed_count': 7266103},{'id': 1738,'hero': 'https://pe.tedcdn.com/images/ted/de98b161ad1434910ff4b56c89de71af04b8b873_1600x1200.jpg','speaker': 'Ken Robinson',"'title': """"How to escape education's death valley""""",'duration': 1151,'slug': 'ken_robinson_how_to_escape_education_s_death_valley','viewed_count': 6657572},{'id': 2276,'hero': 'https://pe.tedcdn.com/images/ted/3821f3728e0b755c7b9aea2e69cc093eca41abe1_2880x1620.jpg','speaker': 'Linda Cliatt-Wayman','title': 'How to fix a broken school? Lead fearlessly,love hard','duration': 1027,'slug': 'linda_cliatt_wayman_how_to_fix_a_broken_school_lead_fearlessly_love_hard','viewed_count': 1617101},{'id': 892,'hero': 'https://pe.tedcdn.com/images/ted/e79958940573cc610ccb583619a54866c41ef303_2880x1620.jpg','speaker': 'Charles Leadbeater','title': 'Education innovation in the slums','duration': 1138,'slug': 'charles_leadbeater_on_education','viewed_count': 772296},{'id': 1232,'hero': 'https://pe.tedcdn.com/images/ted/0e3e4e92d5ee8ae0e43962d447d3f790b31099b8_800x600.jpg','speaker': 'Geoff Mulgan','title': 'A short intro to the Studio School','duration': 376,'slug': 'geoff_mulgan_a_short_intro_to_the_studio_school','viewed_count': 667971},{'id': 2616,'hero': 'https://pe.tedcdn.com/images/ted/71c...,'speaker': 'Kandice Sumner',"'title': """"How America's public schools keep ...",'duration': 830,'slug': 'kandice_sumner_how_america_s_public_...,"'viewed_count': 1181333}]""",Author/educator,"""['children'",'creativity','culture','dance','education','parenting',"'teaching']""",Do schools kill creativity?,"""https://www.ted.com/talks/ken_robinson_says_s..."
672,"""Tony Robbins discusses the """"invisible forces"""" that motivate everyone's actions -- and high-fives Al Gore in the front row.""",1305,TED2006,1138838400,36,Tony Robbins,Tony Robbins: Why we do what we do,1,1151440680,"""[{'id': 7",'name': 'Funny','count': 1102},{'id': 3,'name': 'Courageous','count': 721},{'id': 21,'name': 'Unconvincing','count': 377},{'id': 2,'name': 'Confusing','count': 301},{'id': 1,'name': 'Beautiful','count': 706},{'id': 9,'name': 'Ingenious','count': 397},{'id': 11,'name': 'Longwinded','count': 285},{'id': 8,'name': 'Informative','count': 1038},{'id': 10,'name': 'Inspiring','count': 5211},{'id': 22,'name': 'Fascinating','count': 1350},{'id': 24,'name': 'Persuasive','count': 2423},{'id': 25,'name': 'OK','count': 441},{'id': 23,'name': 'Jaw-dropping','count': 669},{'id': 26,'name': 'Obnoxious',"'count': 335}]""","""[{'id': 229",'hero': 'https://pe.tedcdn.com/images/ted/eefe30d20338d800bdc70a09dc0f6007e7355a74_2880x1620.jpg','speaker': 'Jill Bolte Taylor','title': 'My 

In [8]:
interactions=pd.read_csv('/content/07_user_interaction_data.csv')

print(interactions.shape)

interactions.head()

(29, 7)


,user_id,ted_talk_id,talk_title,rating,liked,watched,saved
0,U001,TED_SAMPLE_001,Do schools kill creativity?,5.0,True,True,True
1,U001,TED_SAMPLE_002,Averting the climate crisis,4.0,True,True,True
2,U001,TED_SAMPLE_003,Simplicity sells,5.0,True,True,True
3,U001,TED_SAMPLE_004,Greening the ghetto,2.0,False,True,False
4,U001,TED_SAMPLE_005,The best stats you've ever seen,1.0,False,True,False


Prepare data and build models

We prepare the talk content, match interaction records, and create the collaborative filtering model.

In [17]:
# CELL: Clean data and build both recommendation models

import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# 1. Clean column names
talks.columns = talks.columns.str.strip().str.lower().str.replace('"', '', regex=False)
interactions.columns = interactions.columns.str.strip().str.lower().str.replace('"', '', regex=False)


# 2. Clean text values
def clean_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip().strip('"').strip("'")


for col in talks.select_dtypes(include="object").columns:
    talks[col] = talks[col].apply(clean_text)

for col in interactions.select_dtypes(include="object").columns:
    interactions[col] = interactions[col].apply(clean_text)


# 3. Find the required columns
id_col = None

for col in ["ted_talk_id", "talk_id", "id"]:
    if col in talks.columns:
        id_col = col
        break

title_col = None

for col in ["title", "talk_title"]:
    if col in talks.columns:
        title_col = col
        break

interaction_id_col = None

for col in ["ted_talk_id", "talk_id", "id"]:
    if col in interactions.columns:
        interaction_id_col = col
        break

if title_col is None:
    raise ValueError("Talk title column is missing in the TED Talks dataset.")

if "user_id" not in interactions.columns:
    raise ValueError("user_id column is missing in the interactions dataset.")

if "rating" not in interactions.columns:
    raise ValueError("rating column is missing in the interactions dataset.")

if interaction_id_col is None and "talk_title" not in interactions.columns:
    raise ValueError("Interactions must contain a talk ID or talk_title column.")


# 4. Create a clean title for matching
def normalize_title(value):
    value = clean_text(value).lower()
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return " ".join(value.split())


talks["title_clean"] = talks[title_col].apply(normalize_title)

# Remove rows without titles
talks = talks[talks["title_clean"] != ""].copy()

# Remove duplicate TED Talk IDs/titles
talks = talks.drop_duplicates(
    subset=[id_col] if id_col else ["title_clean"]
).copy()


# 5. Create unique talk keys
def normalize_id(value):
    value = clean_text(value)

    if value.endswith(".0"):
        value = value[:-2]

    return value


if id_col:
    talks["talk_key"] = talks[id_col].apply(normalize_id)

    # Use title when the ID is missing
    missing_id = talks["talk_key"] == ""
    talks.loc[missing_id, "talk_key"] = (
        "TITLE_" + talks.loc[missing_id, "title_clean"]
    )
else:
    talks["talk_key"] = "TITLE_" + talks["title_clean"]


# 6. Match interaction records with TED Talks
talk_id_map = {
    key: key for key in talks["talk_key"]
}

title_map = {}

for _, row in talks.iterrows():
    title = row["title_clean"]

    # Only use titles that are unique
    if title not in title_map:
        title_map[title] = row["talk_key"]
    else:
        title_map[title] = None


def find_talk_key(row):

    # First try matching by ID
    if interaction_id_col:
        talk_id = normalize_id(row[interaction_id_col])

        if talk_id in talk_id_map:
            return talk_id

    # If ID does not match, try matching by title
    if "talk_title" in interactions.columns:
        title = normalize_title(row["talk_title"])

        if title in title_map:
            return title_map[title]

    return None


interactions["talk_key"] = interactions.apply(find_talk_key, axis=1)


# 7. Keep only successfully matched records
before_matching = len(interactions)

interactions = interactions[
    interactions["talk_key"].notna()
].copy()

matched_count = len(interactions)
unmatched_count = before_matching - matched_count


# 8. Clean ratings
interactions["rating"] = pd.to_numeric(
    interactions["rating"], errors="coerce"
)

interactions.loc[
    ~interactions["rating"].between(1, 5),
    "rating"
] = np.nan

# Remove duplicate user-talk interactions
interactions = interactions.drop_duplicates(
    subset=["user_id", "talk_key"],
    keep="last"
)


# 9. Prepare text for the content-based model
text_columns = []

for col in [title_col, "tags", "description", "transcript"]:
    if col in talks.columns and col not in text_columns:
        text_columns.append(col)

talks["content"] = talks[text_columns].fillna("").agg(
    " ".join, axis=1
)

if talks["content"].str.strip().eq("").all():
    raise ValueError("No text content available to build the content model.")


# 10. Create the content-based model
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

try:
    content_matrix = tfidf.fit_transform(talks["content"])
except ValueError:
    # Retry without removing English stop words
    tfidf = TfidfVectorizer(max_features=5000)
    content_matrix = tfidf.fit_transform(talks["content"])


# 11. Create the collaborative filtering model
rating_matrix = interactions.pivot_table(
    index="user_id",
    columns="talk_key",
    values="rating",
    aggfunc="mean"
)

# Replace missing ratings with 0 for similarity calculation
rating_matrix_filled = rating_matrix.fillna(0)


# 12. Calculate user similarity
if rating_matrix_filled.shape[0] > 0 and rating_matrix_filled.shape[1] > 0:

    similarity_matrix = pd.DataFrame(
        cosine_similarity(rating_matrix_filled),
        index=rating_matrix.index,
        columns=rating_matrix.index
    )

else:
    similarity_matrix = pd.DataFrame()


# 13. Display results
print("DATA CLEANING AND MODEL CREATION COMPLETED")
print("------------------------------------------")

print("Total TED Talks:", len(talks))
print("Total interactions before matching:", before_matching)
print("Successfully matched interactions:", matched_count)
print("Unmatched interactions:", unmatched_count)
print("Final interaction records:", len(interactions))

print("Content matrix shape:", content_matrix.shape)
print("Rating matrix shape:", rating_matrix.shape)
print("Similarity matrix shape:", similarity_matrix.shape)

if similarity_matrix.empty:
    print("Warning: Collaborative similarity could not be calculated.")
    print("Check whether the interaction data contains valid ratings.")
else:
    print("Both recommendation models are ready!")

DATA CLEANING AND MODEL CREATION COMPLETED
------------------------------------------
Total TED Talks: 397
Total interactions before matching: 0
Successfully matched interactions: 0
Unmatched interactions: 0
Final interaction records: 0
Content matrix shape: (397, 1515)
Rating matrix shape: (0, 0)
Similarity matrix shape: (0, 0)
Check whether the interaction data contains valid ratings.


Split training and testing data

We hide one relevant talk per eligible user. The hidden talk is used to check whether the models can recommend it.

A relevant talk means a talk rated 4 or 5, or marked as liked.

In [20]:
# Find positive interactions
if "liked" in interactions.columns:
    liked = interactions["liked"].astype(str).str.lower().isin(
        ["true", "1", "yes"]
    )
else:
    liked = pd.Series(False, index=interactions.index)

positive = interactions[
    (interactions["rating"] >= 4) | liked
]

# Select one relevant talk per eligible user
test_ids = []

for user in interactions["user_id"].unique():

    user_data = interactions[interactions["user_id"] == user]
    user_positive = positive[positive["user_id"] == user]

    if len(user_data) >= 3 and len(user_positive) > 0:
        test_ids.append(user_positive.index[-1])

if not test_ids:
    raise ValueError("No eligible test interactions found.")

# Create training and testing datasets
test_data = interactions.loc[test_ids].copy()
train_data = interactions.drop(test_ids).copy()

# Store hidden relevant talks
relevant = {}

for user in test_data["user_id"].unique():
    relevant[user] = set(
        test_data[test_data["user_id"] == user]["talk_key"]
    )

# Rebuild collaborative model using training data only
rating_matrix = train_data.pivot_table(
    index="user_id", columns="talk_key", values="rating"
)

similarity_matrix = pd.DataFrame(
    cosine_similarity(rating_matrix.fillna(0)),
    index=rating_matrix.index,
    columns=rating_matrix.index
)

print("Training records:", len(train_data))
print("Testing records:", len(test_data))
print("Users tested:", len(relevant))

Training records: 24
Testing records: 5
Users tested: 5


Create three recommendation functions

All three models use only training data. The hybrid model combines normalized scores.

In [21]:
# CONTENT-BASED RECOMMENDATION
def content_recommend(user, k=5):

    user_data = train_data[train_data["user_id"] == user]

    liked = user_data[
        (user_data["rating"] >= 4) |
        (user_data["liked"].astype(str).str.lower() == "true")
    ]

    if liked.empty:
        return pd.DataFrame(columns=["talk_key", "score"])

    positions = talks.index.get_indexer(
        talks[talks["talk_key"].isin(liked["talk_key"])].index
    )

    profile = content_matrix[positions].mean(axis=0)

    scores = cosine_similarity(
        np.asarray(profile), content_matrix
    ).flatten()

    result = pd.DataFrame({
        "talk_key": talks["talk_key"],
        "score": scores
    })

    seen = user_data["talk_key"].tolist()
    result = result[~result["talk_key"].isin(seen)]

    return result.sort_values("score", ascending=False).head(k)


# COLLABORATIVE RECOMMENDATION
def collaborative_recommend(user, k=5):

    if user not in rating_matrix.index:
        return pd.DataFrame(columns=["talk_key", "score"])

    user_ratings = rating_matrix.loc[user]
    similar_users = similarity_matrix.loc[user].drop(user)

    seen = train_data[
        train_data["user_id"] == user
    ]["talk_key"].tolist()

    results = []

    for talk in talks["talk_key"]:

        if talk in seen:
            continue

        total = 0
        weight = 0

        if talk in rating_matrix.columns:

            for other in similar_users.index:

                similarity = similar_users[other]
                rating = rating_matrix.loc[other, talk]

                if similarity > 0 and pd.notna(rating):
                    total += similarity * rating
                    weight += similarity

        if weight > 0:
            results.append([talk, total / weight / 5])

    return pd.DataFrame(
        results, columns=["talk_key", "score"]
    ).sort_values("score", ascending=False).head(k)


# HYBRID RECOMMENDATION
def hybrid_recommend(user, k=5, content_weight=0.5):

    content = content_recommend(user, 20)
    collaborative = collaborative_recommend(user, 20)

    result = pd.merge(
        content.rename(columns={"score": "content_score"}),
        collaborative.rename(columns={"score": "collab_score"}),
        on="talk_key",
        how="outer"
    )

    if result.empty:
        return result

    result = result.fillna(0)

    result["hybrid_score"] = (
        content_weight * result["content_score"] +
        (1 - content_weight) * result["collab_score"]
    )

    result = result.drop_duplicates("talk_key")

    result = result.merge(
        talks[["talk_key", title_col]],
        on="talk_key",
        how="left"
    )

    return result.sort_values(
        "hybrid_score", ascending=False
    ).head(k)

Evaluate models using Precision and Recall

Understand these two measurements:
- Precision = Relevant recommended talks / 5.
- Recall= Relevant recommended talks / Total relevant hidden talks.

In [22]:
K = 5

# Calculate evaluation scores
def evaluate(recommendations, actual):

    recommended = set(recommendations["talk_key"])
    actual = set(actual)

    correct = len(recommended & actual)

    precision = correct / K
    recall = correct / len(actual) if actual else 0

    return precision, recall


results = []

for user in relevant:

    # Test content-based and collaborative models
    models = {
        "Content": content_recommend(user, K),
        "Collaborative": collaborative_recommend(user, K)
    }

    # Test different hybrid weights
    for weight in [0.25, 0.50, 0.75]:
        models[f"Hybrid {int(weight * 100)}%"] = hybrid_recommend(
            user, K, weight
        )

    # Calculate scores
    for name, recommendations in models.items():

        p, r = evaluate(recommendations, relevant[user])

        f1 = 2 * p * r / (p + r) if p + r > 0 else 0

        results.append([user, name, p, r, f1])


# Create evaluation table
evaluation = pd.DataFrame(
    results,
    columns=["User", "Model", "Precision@5", "Recall@5", "F1 Score"]
)

display(evaluation)

,User,Model,Precision@5,Recall@5,F1 Score
0,U001,Content,0.0,0.0,0.000000
1,U001,Collaborative,0.2,1.0,0.333333
2,U001,Hybrid 25%,0.2,1.0,0.333333
3,U001,Hybrid 50%,0.2,1.0,0.333333
4,U001,Hybrid 75%,0.0,0.0,0.000000
5,U002,Content,0.0,0.0,0.000000
6,U002,Collaborative,0.2,1.0,0.333333
7,U002,Hybrid 25%,0.2,1.0,0.333333
8,U002,Hybrid 50%,0.0,0.0,0.000000
9,U002,Hybrid 75%,0.0,0.0,0.000000


Compare results and select the final model

In [23]:
# Calculate average performance
comparison = evaluation.groupby("Model")[
    ["Precision@5", "Recall@5", "F1 Score"]
].mean().reset_index()

comparison = comparison.sort_values(
    "F1 Score", ascending=False
)

print("MODEL COMPARISON")
display(comparison)

# Select best model
best = comparison.iloc[0]

print("FINAL MODEL:", best["Model"])
print("Precision:", round(best["Precision@5"], 3))
print("Recall:", round(best["Recall@5"], 3))
print("F1 Score:", round(best["F1 Score"], 3))

# Show final recommendations
sample_user = list(relevant.keys())[0]

print("\nCONTENT-BASED")
display(content_recommend(sample_user))

print("\nCOLLABORATIVE")
display(collaborative_recommend(sample_user))

print("\nHYBRID")
display(hybrid_recommend(sample_user))

MODEL COMPARISON


,Model,Precision@5,Recall@5,F1 Score
0,Collaborative,0.20,1.0,0.333333
2,Hybrid 25%,0.20,1.0,0.333333
3,Hybrid 50%,0.12,0.6,0.200000
1,Content,0.00,0.0,0.000000
4,Hybrid 75%,0.00,0.0,0.000000


FINAL MODEL: Collaborative
Precision: 0.2
Recall: 1.0
F1 Score: 0.333

CONTENT-BASED


,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,talk_key,score
578,Why do people succeed? Is it because they're smart? Or are they just lucky? Neither. Analyst Richard St. John condenses years of interviews into an unmissable 3-minute slideshow on the real secrets of success.,210,TED2005,1109116800,66,Richard St. John,Richard St. John: 8 secrets of success,1,1166055060,"""[{'id': 7",'name': 'Funny','count': 2966},{'id': 1,'name': 'Beautiful','count': 576},{'id': 21,'name': 'Unconvincing','count': 213},{'id': 8,'name': 'Informative','count': 1912},{'id': 10,'name': 'Inspiring','count': 4978},{'id': 2,'name': 'Confusing','count': 43},{'id': 22,'name': 'Fascinating','count': 777},{'id': 9,'name': 'Ingenious','count': 638},{'id': 3,'name': 'Courageous','count': 457},{'id': 11,'name': 'Longwinded','count': 27},{'id': 24,'name': 'Persuasive','count': 1433},{'id': 23,'name': 'Jaw-dropping','count': 251},{'id': 25,'name': 'OK','count': 685},{'id': 26,'name': 'Obnoxious',"'count': 57}]""","""[{'id': 66",'hero': 'https://pe.tedcdn.com/images/ted/6b6eb940bceab359ca676a9b486aae475c1df883_2880x1620.jpg','speaker': 'Ken Robinson','title': 'Do schools kill creativity?','duration': 1164,'slug': 'ken_robinson_says_schools_kill_creativity','viewed_count': 47227171},{'id': 202,'hero': 'https://pe.tedcdn.com/images/ted/87205c20e5b3e1b63d04da23c04b22956087f0be_2880x1620.jpg','speaker': 'Gever Tulley','title': '5 dangerous things you should let your kids do','duration': 558,'slug': 'gever_tulley_on_5_dangerous_things_for_kids','viewed_count': 4364853},{'id': 93,'hero': 'https://pe.tedcdn.com/images/ted/bf2541157cdf5741f11ddea013c19b4ce5201be4_2880x1620.jpg','speaker': 'Barry Schwartz','title': 'The paradox of choice','duration': 1177,'slug': 'barry_schwartz_on_the_paradox_of_choice','viewed_count': 10000711},{'id': 572,'hero': 'https://pe.tedcdn.com/images/ted/6c5a57b8edbb8082ef49b64994c3a1957f5e13a9_1600x1200.jpg','speaker': 'Richard St. John','title': 'Success is a continuous journey','duration': 237,'slug': 'richard_st_john_success_is_a_continuous_journey','viewed_count': 3661589},{'id': 2638,'hero': 'https://pe.tedcdn.com/images/ted/f3b1cafe89d952edc479031dbd2df69faf730d74_2880x1620.jpg','speaker': 'Jia Jiang','title': 'What I learned from 100 days of rejection','duration': 931,'slug': 'jia_jiang_what_i_learned_from_100_days_of_rejection','viewed_count': 3464229},TITLE_success,0.423789
86,Graphic designer Stefan Sagmeister takes the audience on a whimsical journey through moments of his life that made him happy -- and notes how many of these moments have to do with good design.,930,TED2004,1075680000,26,Stefan Sagmeister,Stefan Sagmeister: Happiness by design,1,1175731860,"""[{'id': 2",'name': 'Confusing','count': 31},{'id': 7,'name': 'Funny','count': 468},{'id': 9,'name': 'Ingenious','count': 159},{'id': 1,'name': 'Beautiful','count': 180},{'id': 11,'name': 'Longwinded','count': 68},{'id': 8,'name': 'Informative','count': 100},{'id': 10,'name': 'Inspiring','count': 542},{'id': 22,'name': 'Fascinating','count': 138},{'id': 3,'name': 'Courageous','count': 52},{'id': 21,'name': 'Unconvincing','count': 70},{'id': 25,'name': 'OK','count': 123},{'id': 26,'name': 'Obnoxious','count': 14},{'id': 23,'name': 'Jaw-dropping','count': 23},{'id': 24,'name': 'Persuasive',"'count': 33}]""","""[{'id': 356",'hero': 'https://pe.tedcdn.com/images/ted/639de4f62f2fa9419737f97dd6078b014f7e4b0a_2880x1620.jpg','speaker': 'Stefan Sagmeister',"'title': """"Things I've learned in my life so far""""",'duration': 285,'slug': 'stefan_sagmeister_on_what_he_has_learned','viewed_count': 724824},{'id': 182,'hero': 'https://pe.tedcdn.com/images/ted/b8975cef603ff7110ea49f95c3e197866b5c397f_1600x1200.jpg','speaker': 'Maira Kalman','title': 'The illustrated woman','duration': 1050,'slug': 'maira_kalman_the_illustrated_woman','viewed_count': 672251},{'id': 172,'hero': 'https://pe.tedcdn.com/images/ted/b790be2f87ceffba73fe73837944400c7d61cba2_16


COLLABORATIVE


,talk_key,score
0,TITLE_library,0.592242
1,TITLE_meet the future of cancer research,0.200000



HYBRID


,talk_key,content_score,collab_score,hybrid_score,title
12,TITLE_library,0.000000,0.592242,0.296121,library']
19,TITLE_success,0.423789,0.000000,0.211894,success
11,TITLE_https www ted com talks stefan sagmeiste...,0.350473,0.000000,0.175237,https://www.ted.com/talks/stefan_sagmeister_sh...
16,TITLE_poetry,0.343925,0.000000,0.171962,poetry
7,TITLE_https www ted com talks nathalie miebach,0.343582,0.000000,0.171791,https://www.ted.com/talks/nathalie_miebach
